In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import cv2, random, os
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import normalize
import math
import seaborn as sns
import numpy as np
import torchvision.transforms as T

In [33]:
train_df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/train.csv")
sub_df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/sample_submission.csv")

In [34]:
TRAIN_DIR = Path("/kaggle/input/competitions/landmark-recognition-2021/train")
TEST_DIR  = Path("/kaggle/input/competitions/landmark-recognition-2021/test")

In [35]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True

In [36]:
def make_subset(df, frac=0.05, seed=42):
    df = df.sample(frac=frac, random_state=seed).reset_index(drop=True)
    return df

def split_train_val_by_class(df, val_ratio=0.2, seed=42):
    train_parts = []
    val_parts = []

    rng = np.random.RandomState(seed)

    for cls, g in df.groupby("landmark_id"):
        g = g.sample(frac=1, random_state=seed).reset_index(drop=True)
        n = len(g)

        if n == 1:
            train_parts.append(g)
            continue

        n_val = max(1, int(round(n * val_ratio)))
        if n_val >= n:
            n_val = n - 1

        val_idx = rng.choice(n, size=n_val, replace=False)
        val_mask = np.zeros(n, dtype=bool)
        val_mask[val_idx] = True

        val_parts.append(g[val_mask])
        train_parts.append(g[~val_mask])

    train_df = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_parts).sample(frac=1, random_state=seed).reset_index(drop=True) if val_parts else pd.DataFrame(columns=df.columns)

    return train_df, val_df

In [37]:
DEBUG = True

CFG = {
    "backbone": "resnet18",
    "embedding_size": 256,
    "batch_size": 8,
    "image_size": 224,
    "epochs": 2,
    "lr": 1e-4,
    "margin": 0.3,
    "scale": 20.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

In [38]:
if DEBUG:
    subset_frac = 0.05
    CFG["batch_size"] = 8
    CFG["image_size"] = 224
    CFG["epochs"] = 2
else:
    subset_frac = 1.0
    CFG["batch_size"] = 16
    CFG["image_size"] = 384
    CFG["epochs"] = 5

In [39]:
def make_debug_df(df, n_classes=8, per_class=4, seed=42):
    parts = []
    classes = df["landmark_id"].value_counts().head(n_classes).index.tolist()

    for cls in classes:
        part = df[df["landmark_id"] == cls].sample(
            n=min(per_class, (df["landmark_id"] == cls).sum()),
            random_state=seed
        )
        parts.append(part)

    out = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

In [40]:
if DEBUG:
    train_df = make_debug_df(train_df)

In [41]:
class LandmarkDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def load_image(self, img_id):
        path = self.img_dir / img_id[0] / img_id[1] / img_id[2] / f"{img_id}.jpg"
        img = Image.open(path).convert("RGB")
        return ImageOps.exif_transpose(img)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.load_image(row["id"])
        if self.transform:
            img = self.transform(img)
        label = int(row["label_idx"])
        return img, torch.tensor(label, dtype=torch.long)

In [ ]:
train_tfms = T.Compose([
    T.Resize((CFG["image_size"], CFG["image_size"])),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tfms = T.Compose([
    T.Resize((CFG["image_size"], CFG["image_size"])),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [43]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * float(p))
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=(-1, -2)).pow(1.0 / self.p)

In [44]:
class EmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG["backbone"],
            pretrained=True,
            num_classes=0
        )
        self.pool = GeM(3.0)
        self.fc = nn.Linear(self.backbone.num_features, CFG["embedding_size"])
        self.bn = nn.BatchNorm1d(CFG["embedding_size"])
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.pool(x)
        x = self.fc(x)
        x = self.dropout(x)
        x = self.bn(x)
        return F.normalize(x, dim=1)

In [45]:
class ArcFaceLoss(nn.Module):
    def __init__(self, num_classes, embedding_size, margin=0.3, scale=30.0):
        super().__init__()
        self.margin = margin
        self.scale = scale

        self.W = nn.Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.W)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings, labels):
        W = F.normalize(self.W)
        cosine = F.linear(embeddings, W)
        

        sine = torch.sqrt((1.0 - cosine**2).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1)

        logits = one_hot * phi + (1 - one_hot) * cosine
        logits *= self.scale

        return F.cross_entropy(logits, labels)

In [46]:
full_df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/train.csv")
subset_df = make_subset(full_df, frac=0.5, seed=42)
train_df, val_df = split_train_val_by_class(subset_df, val_ratio=0.2, seed=42)

top_classes = train_df["landmark_id"].value_counts().head(10000).index

train_df = train_df[train_df["landmark_id"].isin(top_classes)]
val_df   = val_df[val_df["landmark_id"].isin(top_classes)]

unique_labels = sorted(train_df["landmark_id"].unique())

label2idx = {l: i for i, l in enumerate(unique_labels)}
idx2label = {i: l for l, i in label2idx.items()}

train_df["label_idx"] = train_df["landmark_id"].map(label2idx)

val_df = val_df[val_df["landmark_id"].isin(label2idx)].copy()
val_df["label_idx"] = val_df["landmark_id"].map(label2idx)

print("Train min/max:", train_df["label_idx"].min(), train_df["label_idx"].max())
print("Val min/max:", val_df["label_idx"].min(), val_df["label_idx"].max())
print("Num classes:", len(label2idx))

Train min/max: 0 9999
Val min/max: 0 9999
Num classes: 10000


In [47]:
train_ds = LandmarkDataset(train_df, TRAIN_DIR, transform=train_tfms)
val_ds   = LandmarkDataset(val_df, TRAIN_DIR, transform=val_tfms)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=pin_memory,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory,
)

In [48]:
# @torch.no_grad()
# def evaluate(model, criterion, loader):
#     model.eval()
#     criterion.eval()

#     total_loss = 0.0
#     total = 0
#     correct = 0

#     W = F.normalize(criterion.W)

#     for imgs, labels in loader:
#         imgs = imgs.to(CFG["device"], non_blocking=True)
#         labels = labels.to(CFG["device"], non_blocking=True)

#         with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
#             embeddings = model(imgs)
#             loss = criterion(embeddings, labels)

#             cosine_logits = F.linear(embeddings, W) * CFG["scale"]
#             preds = cosine_logits.argmax(dim=1)

#         bs = labels.size(0)
#         total_loss += loss.item() * bs
#         total += bs
#         correct += (preds == labels).sum().item()

#     return total_loss / max(total, 1), correct / max(total, 1)

In [49]:
def average_precision(gt_label, preds):
    for i, (pred_label, _) in enumerate(preds, 1):
        if pred_label == gt_label:
            return 1.0 / i
    return 0.0

In [50]:
def compute_gap(prediction_strings, gt_labels):
    ap_sum = 0.0

    for pred_str, gt in zip(prediction_strings, gt_labels):
        if not pred_str.strip():
            ap = 0.0
        else:
            parts = pred_str.strip().split()
            pred_list = []

            for p in range(0, len(parts), 2):
                lbl = int(parts[p])
                conf = float(parts[p+1])
                pred_list.append((lbl, conf))

            pred_list.sort(key=lambda x: x[1], reverse=True)
            ap = average_precision(gt, pred_list)

        ap_sum += ap

    return ap_sum / len(gt_labels)

In [51]:
def get_prediction_strings(logits, topk=5):
    probs = logits.softmax(dim=1)
    topk_vals, topk_idx = probs.topk(topk, dim=1)

    results = []

    for vals, idxs in zip(topk_vals, topk_idx):
        parts = []
        for v, i in zip(vals, idxs):
            parts.append(f"{int(i)} {float(v)}")
        results.append(" ".join(parts))

    return results

In [52]:
@torch.no_grad()
def get_embeddings(model, loader):
    model.eval()
    
    all_embs = []
    all_labels = []
    
    for imgs, labels in loader:
        imgs = imgs.to(CFG["device"])
        
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            emb = model(imgs)
        
        all_embs.append(emb.cpu())
        all_labels.extend(labels.numpy())
    
    all_embs = torch.cat(all_embs, dim=0)  # [N, D]
    all_labels = torch.tensor(all_labels)
    
    return all_embs, all_labels

In [53]:
@torch.no_grad()
def evaluate_gap_knn(model, train_loader, val_loader, topk=5):
    model.eval()

    train_embs, train_labels = get_embeddings(model, train_loader)
    train_embs = F.normalize(train_embs, dim=1)
    train_labels_np = train_labels.cpu().numpy()

    all_preds = []
    all_gts = []

    for imgs, labels in val_loader:
        imgs = imgs.to(CFG["device"])
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            val_embs = model(imgs).cpu()
        val_embs = F.normalize(val_embs, dim=1)

        sim = val_embs @ train_embs.T

        topk_vals, topk_idx = sim.topk(topk, dim=1)

        for vals, idxs, gt in zip(topk_vals, topk_idx, labels):
            unique = {}
            for v, idx in zip(vals, idxs):
                lbl = int(train_labels_np[idx])
                conf = float(v)
                if lbl not in unique or conf > unique[lbl]:
                    unique[lbl] = conf

            sorted_items = sorted(unique.items(), key=lambda x: x[1], reverse=True)[:topk]
            parts = [f"{lbl} {conf}" for lbl, conf in sorted_items]
            all_preds.append(" ".join(parts))
            all_gts.append(gt.item())

    for i in range(min(5, len(all_preds))):
        print(f"\n--- Sample {i} (GT={all_gts[i]}) ---")
        parts = all_preds[i].split()
        for p in range(0, len(parts), 2):
            print(f"  Pred: {parts[p]} conf={float(parts[p+1]):.4f}")

    return compute_gap(all_preds, all_gts)

In [54]:
CFG["image_size"] = 128
CFG["batch_size"] = 16

train_tfms = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

In [55]:
CFG["epochs"] = 10

In [56]:
CFG["backbone"] = "tf_efficientnet_b3"

In [ ]:
model = EmbeddingNet().to(CFG["device"])
criterion = ArcFaceLoss(
    num_classes=len(label2idx),
    embedding_size=CFG["embedding_size"],
    margin=CFG["margin"],
    scale=CFG["scale"]
).to(CFG["device"])

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()),
    lr=CFG["lr"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

imgs, labels = next(iter(train_loader))
imgs = imgs.to(CFG["device"])
labels = labels.to(CFG["device"])

with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
    emb = model(imgs)
    loss = criterion(emb, labels)

if torch.isnan(loss):
    print("NaN detected!")
    raise ValueError("NaN in loss")

print("Sanity check OK:", emb.shape, loss.item())

best_val_gap = 0.0

for epoch in range(CFG["epochs"]):
    model.train()
    criterion.train()

    total_loss = 0.0
    total = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(CFG["device"], non_blocking=True)
        labels = labels.to(CFG["device"], non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            embeddings = model(imgs)
            loss = criterion(embeddings, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total += bs

    train_loss = total_loss / max(total, 1)

    val_gap = evaluate_gap_knn(model, train_loader, val_loader, topk=5)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_gap={val_gap:.4f}")

    if val_gap > best_val_gap:
        best_val_gap = val_gap
        torch.save({
            "model": model.state_dict(),
            "criterion": criterion.state_dict(),
            "label2idx": label2idx,
            "idx2label": idx2label,
            "cfg": CFG,
        }, "/kaggle/working/best_arcface_debug.pth")
        print("Saved best checkpoint")
        
    scheduler.step()

Sanity check OK: torch.Size([16, 256]) 16.160322189331055

--- Sample 0 (GT=644) ---
  Pred: 3834 conf=0.9465
  Pred: 644 conf=0.9401
  Pred: 5046 conf=0.9264
  Pred: 8851 conf=0.9247

--- Sample 1 (GT=3959) ---
  Pred: 3959 conf=0.8741

--- Sample 2 (GT=4650) ---
  Pred: 7307 conf=0.8914
  Pred: 4807 conf=0.8433
  Pred: 1512 conf=0.8262
  Pred: 3080 conf=0.8175

--- Sample 3 (GT=251) ---
  Pred: 251 conf=0.9867
  Pred: 6933 conf=0.9748
  Pred: 1241 conf=0.9669
  Pred: 1069 conf=0.9667
  Pred: 4745 conf=0.9661

--- Sample 4 (GT=5016) ---
  Pred: 6406 conf=0.7417
  Pred: 2856 conf=0.7210
  Pred: 4231 conf=0.6899
  Pred: 3455 conf=0.6870
Epoch 1: train_loss=11.6403 val_gap=0.7290
Saved best checkpoint

--- Sample 0 (GT=644) ---
  Pred: 644 conf=0.8836
  Pred: 7670 conf=0.8341
  Pred: 5046 conf=0.8031

--- Sample 1 (GT=3959) ---
  Pred: 3959 conf=0.9324

--- Sample 2 (GT=4650) ---
  Pred: 4650 conf=0.7581
  Pred: 9404 conf=0.7447
  Pred: 3584 conf=0.7447
  Pred: 7307 conf=0.7423

--- Samp